## Setup (Dependencies & Environment)

**Python**: 3.12+

**Install**:

```bash
uv add openai python-dotenv ipykernel
uv sync
```

or

```bash
pip install openai python-dotenv ipykernel
```

**Environment**:

Set `OPENAI_API_KEY` in your shell or create a `.env` file in the same folder (this notebook calls `load_dotenv(override=True)`).

Example `.env`:

```bash
OPENAI_API_KEY=your_key_here
```


# Embeddings

This notebook demonstrates semantic closeness using cosine similarity.
Words with similar meanings should have higher similarity scores than unrelated ones.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True) #.env values has the priority. #If an environment variable already exists, replace it with the value from the .env file

True

## Getting embeddings

In [2]:
from openai import OpenAI
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small"
)

print(response.data[0].embedding)

[0.005130767822265625, 0.0171966552734375, -0.0187225341796875, -0.0185546875, -0.047271728515625, -0.030303955078125, 0.027679443359375, 0.0036640167236328125, 0.01123809814453125, 0.00641632080078125, -0.00168609619140625, 0.0158233642578125, -0.0012884140014648438, -0.0078125, 0.05987548828125, 0.050262451171875, -0.0274658203125, 0.009918212890625, -0.04034423828125, 0.049957275390625, -0.0004527568817138672, 0.03021240234375, -0.01372528076171875, 0.03289794921875, 0.017303466796875, 0.01678466796875, -0.0017557144165039062, 0.020416259765625, 0.040771484375, -0.037750244140625, -0.026092529296875, -0.04998779296875, 0.024139404296875, -0.05517578125, -0.0322265625, 0.042388916015625, 0.064697265625, 0.01471710205078125, -0.0156402587890625, -0.04132080078125, 0.02215576171875, 0.007335662841796875, 0.044891357421875, 0.007068634033203125, -0.0240936279296875, 0.052398681640625, -0.02008056640625, -0.03216552734375, 0.0162200927734375, 0.046478271484375, 0.0238494873046875, -0.018

In [3]:
# Dimension
len(response.data[0].embedding)

1536

In [4]:
response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small",
    dimensions=100
)

print(response.data[0].embedding)
print(len(response.data[0].embedding))

[0.0178375244140625, 0.06011962890625, -0.06536865234375, -0.064697265625, -0.1646728515625, -0.1055908203125, 0.096435546875, 0.0127105712890625, 0.039215087890625, 0.0223236083984375, -0.005847930908203125, 0.055145263671875, -0.00437164306640625, -0.0273284912109375, 0.2088623046875, 0.17529296875, -0.095947265625, 0.034637451171875, -0.140625, 0.17431640625, -0.0015010833740234375, 0.10516357421875, -0.04779052734375, 0.1148681640625, 0.06036376953125, 0.058563232421875, -0.006122589111328125, 0.07135009765625, 0.1422119140625, -0.1314697265625, -0.09112548828125, -0.17431640625, 0.08428955078125, -0.1925048828125, -0.1123046875, 0.1475830078125, 0.2254638671875, 0.05133056640625, -0.054656982421875, -0.1441650390625, 0.07720947265625, 0.0258331298828125, 0.1566162109375, 0.02471923828125, -0.083984375, 0.1826171875, -0.07000732421875, -0.1123046875, 0.056732177734375, 0.1619873046875, 0.08306884765625, -0.065673828125, -0.0185394287109375, 0.1533203125, -0.021514892578125, 0.02534

## Cosine similarity

Cosine similarity measures how close two vectors are in direction (ignoring magnitude).
A score of **1.0** means identical direction, **0** means orthogonal, and **−1** means opposite.

In [5]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

In [6]:
# Quick sanity check with simple 2D vectors
cosine_similarity([1, 0], [1, 0])
# cosine_similarity([1, 0], [-1, 0])
#cosine_similarity([1, 0], [0, 1])
#cosine_similarity([1, 0], [1, 1])

1.0

## Comparing word embeddings

Let's embed a set of words and see how cosine similarity captures semantic relationships.

In [7]:
words = [
    "cat", "dog", "kitten", "puppy",
    "car", "automobile",
    "banana", "apple",
    "king", "queen", "man", "woman",
    "Paris", "France", "Tokyo", "Japan", "Singapore",
    "汽车"
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=words,
)

embedding_by_word = {
    word: response.data[i].embedding
    for i, word in enumerate(words)
}

In [8]:
pairs = [
    ("cat", "dog"),
    ("kitten", "puppy"),
    ("car", "automobile"),
    ("king", "queen"),
    ("man", "woman"),
    ("Paris", "France"),
    ("Paris", "Singapore"),
    ("Tokyo", "Japan"),
    ("banana", "apple"),
    ("cat", "banana"),
    ("car", "king"),
    ("Paris", "puppy"),
    ("queen", "automobile"),
    ("汽车", "automobile"),
    ("汽车", "car"),
    ("汽车", "dog"),
]

print("Cosine similarity (higher = more similar):\n")
for a, b in pairs:
    score = cosine_similarity(embedding_by_word[a], embedding_by_word[b])
    print(f"  {a:<12} {b:<12} {score:.3f}")

Cosine similarity (higher = more similar):

  cat          dog          0.603
  kitten       puppy        0.650
  car          automobile   0.556
  king         queen        0.591
  man          woman        0.707
  Paris        France       0.661
  Paris        Singapore    0.399
  Tokyo        Japan        0.471
  banana       apple        0.464
  cat          banana       0.298
  car          king         0.331
  Paris        puppy        0.204
  queen        automobile   0.219
  汽车           automobile   0.679
  汽车           car          0.546
  汽车           dog          0.289
